# Study 4

Tested whether participants chose a point-estimate or range-estimate model for an incentivized slider task.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data and exclude incomplete choices and flagged duplicates.
d = pd.read_excel(ROOT / 'data/raw/Study 4.xlsx')
raw_n = len(d)
d = d.loc[d.DV.notna()].copy()
d = d.loc[(d.ExcludeDuplicateIP != 1) & (d.ExcludeDuplicateID != 1)].copy()
assert len(d) == 392

rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':392, 'match':len(d)==392}]))

,raw_n,replicated_n,paper_n,match
0,446,392,392,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

In [4]:
# Estimate model-choice effects and test the proportion choosing ranges.
assert d.alpha.isin(['range', 'point']).all() and d.DV.isin([1, 2]).all()
d['chose_alpha'] = (d.DV == 1).astype(float)
d['range_first'] = d.alpha.map({'range': 0.5, 'point': -0.5})
record(4, 'Model Alpha: range vs point', ols(d, 'chose_alpha', ['range_first']), 'range_first', 0.41, 0.05)
logit = sm.Logit(d.chose_alpha, sm.add_constant(d[['range_first']])).fit(disp=False)
odds = np.exp(logit.params.range_first)
odds_se = odds * logit.bse.range_first
assert abs(odds - 5.88) <= 0.0050001 and abs(odds_se - 1.31) <= 0.0050001
chose_range = d.chosen == 'range'
test = binomtest(int(chose_range.sum()), len(d), 0.5)
display(pd.DataFrame([{'n': len(d), 'chose_range_n': int(chose_range.sum()), 'chose_range_share': chose_range.mean(), 'paper_range_share': .707, 'binomial_p': test.pvalue, 'logit_odds_ratio': odds, 'paper_odds_ratio': 5.88, 'logit_odds_ratio_se': odds_se, 'paper_odds_ratio_se': 1.31, 'logit_z': logit.tvalues.range_first, 'logit_p': logit.pvalues.range_first}]))
assert round(chose_range.mean() * 100, 1) == 70.7

,n,chose_range_n,chose_range_share,paper_range_share,binomial_p,logit_odds_ratio,paper_odds_ratio,logit_odds_ratio_se,paper_odds_ratio_se,logit_z,logit_p
0,392,277,0.7066,0.707,1.5602e-16,5.88,5.88,1.3132,1.31,7.9322,2.1531e-15


In [5]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,Model Alpha: range vs point,392,0.4129,0.41,0.0459,0.05,1.0998e-17,True,True
